# Day 9 — Practice Session · **SOLUTIONS**### File I/O · Python for Data Science**Prepared by Srinivasa Sai Chava**  ·  Boston University---> **Instructor copy.** Every question is followed by the answer and the reasoning.> The student copy (`Day9_Practice_Questions.ipynb`) is identical minus the answer blocks.| Part | Focus | Questions ||---|---|---|| A | Predict the output | 8 || B | Spot & fix the bug | 4 || C | Write the code | 5 || D | Challenge | 2 |

---## Setup — run this firstCreates the sample files every question below uses.

In [ ]:
import csvwith open("names.txt", "w", encoding="utf-8") as f:    f.write("Ravi\n")    f.write("Sara\n")    f.write("Amit")                # no newline on the last linewith open("marks.csv", "w", newline="", encoding="utf-8") as f:    w = csv.writer(f)    w.writerow(["name", "city", "subject", "mark"])    w.writerows([        ["Ravi", "Pune",       "python", 88],        ["Sara", "Mumbai, MH", "python", 91],        ["Ravi", "Pune",       "stats",  71],        ["Amit", "Delhi",      "stats",  35],    ])print("Sample files ready.")print(open("marks.csv", encoding="utf-8").read())

---## Setup — run this firstCreates the sample files every question below uses.

In [ ]:
import csvwith open("names.txt", "w", encoding="utf-8") as f:    f.write("Ravi\n")    f.write("Sara\n")    f.write("Amit")                # no newline on the last linewith open("marks.csv", "w", newline="", encoding="utf-8") as f:    w = csv.writer(f)    w.writerow(["name", "city", "subject", "mark"])    w.writerows([        ["Ravi", "Pune",       "python", 88],        ["Sara", "Mumbai, MH", "python", 91],        ["Ravi", "Pune",       "stats",  71],        ["Amit", "Delhi",      "stats",  35],    ])print("Sample files ready.")print(open("marks.csv", encoding="utf-8").read())

---# Part A — Predict the Output  *(8 min)*

---# Part A — Predict the Output**Teaching note:** A2 (the trailing newline) and A5 (the exhausted file) are the two thatcause the most confusion in the practice session.

### A1. What is printed?

In [ ]:
with open("names.txt", encoding="utf-8") as f:    print(repr(f.read()))

*Your prediction:*  

> ### ✅ Answer A1> ```> 'Ravi\nSara\nAmit'> ```> **Why:** `read()` returns the **whole file as one string**, newlines included. `repr()`> shows them as `\n` rather than breaking the line, which is exactly why `repr()` is the> right tool when debugging file content.>> Note the last line has **no** trailing `\n` — the file was written that way.

### A2. Which comparisons are True?

In [ ]:
with open("names.txt", encoding="utf-8") as f:    for line in f:        print(repr(line), line == "Ravi", line.strip() == "Ravi")

*Your prediction:*  

> ### ✅ Answer A2> ```> 'Ravi\n'  False  True> 'Sara\n'  False  False> 'Amit'    False  False> ```> **Why:** every line keeps its newline, so `line == "Ravi"` is `False` even for the first> line. Only the stripped comparison works.>> **The asymmetry matters:** the last line has no `\n`, so a program that assumes every> line ends the same way will behave differently on the final record. Always `.strip()`.

### A3. What does the file contain afterwards?

In [ ]:
with open("demo.txt", "w", encoding="utf-8") as f:    f.write("Ravi")    f.write("Sara")print(repr(open("demo.txt", encoding="utf-8").read()))

*Your prediction:*  

> ### ✅ Answer A3> ```> 'RaviSara'> ```> **Why:** `write()` adds **no** newline. It writes exactly the characters you give it.>> To get separate lines you must add them yourself — `f.write(n + "\n")` — or use> `print(n, file=f)`, which does add one.>> `writelines()` does not add newlines either, despite the name.

### A4. What does the file contain now?

In [ ]:
with open("demo.txt", "w", encoding="utf-8") as f:    f.write("first\n")with open("demo.txt", "w", encoding="utf-8") as f:    f.write("second\n")print(repr(open("demo.txt", encoding="utf-8").read()))

*Your prediction:*  

> ### ✅ Answer A4> ```> 'second\n'> ```> **Why:** `"w"` **erases the file completely** the moment it opens. `"first"` is gone —> no warning, no confirmation, no undo.>> Change the second mode to `"a"` and you get `'first\nsecond\n'` instead.>> **This is the only bug on today's list that destroys data.** Everything else merely> wastes an afternoon.

### A5. What does the second read give?

In [ ]:
with open("names.txt", encoding="utf-8") as f:    first  = f.read()    second = f.read()print(len(first), repr(second))

*Your prediction:*  

> ### ✅ Answer A5> ```> 14 ''> ```> **Why:** after the first `read()` the file position sits at the end, so the second call> finds nothing left and returns an empty string.>> **Same idea as Day 6's generator exhaustion** — a one-pass resource. Store what you read,> or open the file again. (`f.seek(0)` also rewinds, but storing is usually clearer.)

### A6. How many fields does each row have?

In [ ]:
with open("marks.csv", encoding="utf-8") as f:    next(f)                              # skip the header    for line in f:        print(len(line.strip().split(",")), line.strip().split(","))

*Your prediction:*  

> ### ✅ Answer A6> ```> 4 ['Ravi', 'Pune', 'python', '88']> 5 ['Sara', '"Mumbai', ' MH"', 'python', '91']> 4 ['Ravi', 'Pune', 'stats', '71']> 4 ['Amit', 'Delhi', 'stats', '35']> ```> **Why:** Sara's city is `"Mumbai, MH"` — a comma **inside** a quoted field. `split(",")`> knows nothing about quoting, so it cuts the city in half and produces five fields instead> of four.>> Every downstream index is then wrong for that one row — which is worse than an error,> because the program keeps running.

### A7. Same file, using the csv module. What changes?

In [ ]:
import csvwith open("marks.csv", newline="", encoding="utf-8") as f:    reader = csv.reader(f)    next(reader)    for row in reader:        print(len(row), row)

*Your prediction:*  

> ### ✅ Answer A7> ```> 4 ['Ravi', 'Pune', 'python', '88']> 4 ['Sara', 'Mumbai, MH', 'python', '91']> 4 ['Ravi', 'Pune', 'stats', '71']> 4 ['Amit', 'Delhi', 'stats', '35']> ```> **Why:** `csv.reader` understands quoting, so Sara's city stays in one piece and every> row has four fields.>> Also notice the marks: `'88'` with quotes. **Every CSV value arrives as a string** —> Day 1's `input()` lesson in a new place. `'88' + 1` is a `TypeError`.

### A8. What type is `row`, and what does it print?

In [ ]:
import csvwith open("marks.csv", newline="", encoding="utf-8") as f:    row = next(csv.DictReader(f))print(type(row).__name__)print(row["name"], row["mark"])

*Your prediction:*  

> ### ✅ Answer A8> ```> dict> Ravi 88> ```> **Why:** `DictReader` uses the header row as keys, so each row comes back as a dictionary> and you address fields by **name** rather than position. It also consumes the header for> you — no `next()` needed to skip it.>> `row["mark"]` is still the string `'88'`. The print looks identical to an integer, which> is exactly why this bug survives to production.

---# Part B — Spot & Fix the Bug  *(7 min)*

---# Part B — Spot & Fix the Bug**Teaching note:** B1 is the destructive one. Make sure everyone understands it beforethey start Part C.

### B1. This should READ the file. What does it actually do?

In [ ]:
# First, put something in the filewith open("important.txt", "w", encoding="utf-8") as f:    f.write("valuable data\n")# Now "read" itwith open("important.txt", "w", encoding="utf-8") as f:    content = f.read()print(repr(open("important.txt", encoding="utf-8").read()))

*What's wrong:* *Your fix:*

> ### ✅ Answer B1> **Symptom:** the file is now empty, and `f.read()` raised `io.UnsupportedOperation`> (or returned nothing) because the file was not opened for reading.> **Cause:** `"w"` erases the file the instant it opens. The data was destroyed before a> single line of the block ran.>> **This is the one file bug that loses work.** There is no undo.

In [ ]:
with open("important.txt", "w", encoding="utf-8") as f:    f.write("valuable data\n")with open("important.txt", "r", encoding="utf-8") as f:    # "r" - or omit it    content = f.read()print(repr(content))print("file intact:", repr(open("important.txt", encoding="utf-8").read()))# "r" is the default, so open(path) alone is safest.# Say the mode out loud before you type it.

### B2. Why does nothing match?

In [ ]:
with open("names.txt", encoding="utf-8") as f:    for line in f:        if line == "Sara":            print("Found Sara")print("done")

*What's wrong:* *Your fix:*

> ### ✅ Answer B2> **Symptom:** prints only `done` — Sara is never found.> **Cause:** the line is `"Sara\n"`, not `"Sara"`. Every line read from a file keeps its> newline character.

In [ ]:
with open("names.txt", encoding="utf-8") as f:    for line in f:        if line.strip() == "Sara":            print("Found Sara")print("done")# .strip() also removes stray spaces, which real files are full of.

### B3. This should total the marks.

In [ ]:
import csvtotal = 0with open("marks.csv", newline="", encoding="utf-8") as f:    for row in csv.DictReader(f):        total += row["mark"]print(total)

*Error:* *Your fix:*

> ### ✅ Answer B3> **Error:** `TypeError: unsupported operand type(s) for +=: 'int' and 'str'`> **Cause:** every CSV value is a **string**. `row["mark"]` is `'88'`, not `88`.

In [ ]:
import csvtotal = 0with open("marks.csv", newline="", encoding="utf-8") as f:    for row in csv.DictReader(f):        total += int(row["mark"])       # cast at the doorprint(total)          # 285# Better still, cast once as you load, so nothing downstream has to remember:rows = []with open("marks.csv", newline="", encoding="utf-8") as f:    for row in csv.DictReader(f):        row["mark"] = int(row["mark"])        rows.append(row)print(sum(r["mark"] for r in rows))

### B4. This crashes if the file is missing.

In [ ]:
with open("no_such_file.csv", encoding="utf-8") as f:    rows = f.readlines()print("loaded", len(rows), "rows")

*Error:* *Your fix:*

> ### ✅ Answer B4> **Error:** `FileNotFoundError: [Errno 2] No such file or directory: 'no_such_file.csv'`> **Cause:** `"r"` mode requires the file to exist, and nothing handles its absence.

In [ ]:
try:    with open("no_such_file.csv", encoding="utf-8") as f:        rows = f.readlines()except FileNotFoundError:    print("No data file — starting empty")    rows = []print("loaded", len(rows), "rows")# Checking first also works:from pathlib import Pathif Path("no_such_file.csv").exists():    print("would read it")else:    print("not there")# But try/except is safer - the file could vanish between the check and the open.

---# Part C — Write the Code  *(12 min)*

---# Part C — Write the Code**Teaching note:** C3 and C5 are the two that matter most — filtering to a new file, andthe full read-process-write cycle.

### C1. Count the linesWrite a function `count_lines(path)` that returns how many lines a file has.Test it on `names.txt`.

In [ ]:
# your code here

In [ ]:
def count_lines(path):    """Return the number of lines in a text file."""    with open(path, encoding="utf-8") as f:        return sum(1 for _ in f)       # counts without loading the whole fileprint(count_lines("names.txt"))        # 3# Also correct, but loads everything into memory:def count_lines_simple(path):    with open(path, encoding="utf-8") as f:        return len(f.readlines())print(count_lines_simple("names.txt"))

### C2. Write a list to a fileWrite `save_names(names, path)` that writes each name on its own line.Then read it back and print it to prove it worked.

In [ ]:
names = ["Ravi", "Sara", "Amit", "Priya"]# your code here

In [ ]:
def save_names(names, path):    """Write each name on its own line."""    with open(path, "w", encoding="utf-8") as f:        for n in names:            f.write(n + "\n")          # write() adds NO newlinenames = ["Ravi", "Sara", "Amit", "Priya"]save_names(names, "saved.txt")print(repr(open("saved.txt", encoding="utf-8").read()))# Equivalent with print():with open("saved2.txt", "w", encoding="utf-8") as f:    for n in names:        print(n, file=f)               # print DOES add the newlineprint(repr(open("saved2.txt", encoding="utf-8").read()))

### C3. Filter a CSV into a new fileRead `marks.csv` and write **only the rows where the mark is 40 or more** into a new filecalled `passed.csv`, keeping the header.Use `DictReader` and `DictWriter`.

In [ ]:
# your code here

In [ ]:
import csvFIELDS = ["name", "city", "subject", "mark"]with open("marks.csv", newline="", encoding="utf-8") as src, \     open("passed.csv", "w", newline="", encoding="utf-8") as dst:    reader = csv.DictReader(src)    writer = csv.DictWriter(dst, fieldnames=FIELDS)    writer.writeheader()    for row in reader:        if int(row["mark"]) >= 40:       # cast before comparing!            writer.writerow(row)print(open("passed.csv", encoding="utf-8").read())# Amit (35) is excluded. Note two files open in ONE with statement,# using a backslash to continue the line.# The input file is never modified - we wrote to a NEW name.

### C4. Append to a logWrite `log(message, path="app.log")` that appends a message on its own line.Call it three times and print the file.

In [ ]:
# your code here

In [ ]:
def log(message, path="app.log"):    """Append one message to the log file."""    with open(path, "a", encoding="utf-8") as f:      # "a" NOT "w"        f.write(message + "\n")log("started")log("loaded 4 rows")log("finished")print(open("app.log", encoding="utf-8").read())# With "w" every call would erase the previous ones and the log would# only ever contain the last line.

### C5. Read, process, writeRead `marks.csv`, compute each **student's** average mark, and write a new CSV`averages.csv` with columns `name` and `average` (one decimal place).

In [ ]:
# your code here

In [ ]:
import csvfrom collections import defaultdict# 1. READ - and cast at the doortotals = defaultdict(list)with open("marks.csv", newline="", encoding="utf-8") as f:    for row in csv.DictReader(f):        totals[row["name"]].append(int(row["mark"]))# 2. PROCESS - Day 6's comprehensionaverages = {name: sum(marks) / len(marks) for name, marks in totals.items()}print(averages)# 3. WRITE - to a NEW filewith open("averages.csv", "w", newline="", encoding="utf-8") as f:    w = csv.writer(f)    w.writerow(["name", "average"])    w.writerows([[n, f"{a:.1f}"] for n, a in averages.items()])print(open("averages.csv", encoding="utf-8").read())# Ravi has two marks (88, 71) -> 79.5# Sara and Amit have one each.

---# Part D — Challenge  *(3 min, or take home)*

---# Part D — Challenge**Teaching note:** D1 combines today with Day 5's counting pattern; D2 is the mergestudents will actually need when they start handling real datasets.

### D1. Word frequency from a fileWrite the text below to `essay.txt`, then read it back and print the **three most commonwords**, ignoring case and full stops.Combine today's file reading with Day 5's counting pattern.

In [ ]:
text = "the cat sat on the mat. the cat was happy. the mat was flat."# your code here

In [ ]:
text = "the cat sat on the mat. the cat was happy. the mat was flat."with open("essay.txt", "w", encoding="utf-8") as f:    f.write(text)def word_frequency(path, top=3):    """Return the most common words in a text file."""    counts = {}    with open(path, encoding="utf-8") as f:        for line in f:            for word in line.lower().replace(".", "").split():                counts[word] = counts.get(word, 0) + 1      # Day 5's pattern    return sorted(counts.items(), key=lambda pair: pair[1], reverse=True)[:top]for word, n in word_frequency("essay.txt"):    print(f"{word}: {n}")# the: 4# cat: 2# mat: 2# Reading line by line means this works on a file of any size - the counts# dictionary is the only thing held in memory.

### D2. Merge two CSV filesCreate a second file `cities.csv` mapping each city to a state, then produce`merged.csv` containing every row of `marks.csv` with a `state` column added.

In [ ]:
# your code here

In [ ]:
import csv# 1. Build the lookup filewith open("cities.csv", "w", newline="", encoding="utf-8") as f:    w = csv.writer(f)    w.writerow(["city", "state"])    w.writerows([["Pune", "Maharashtra"],                 ["Mumbai, MH", "Maharashtra"],                 ["Delhi", "Delhi"]])# 2. Load it into a dict - ONE pass, then fast lookups (Day 5)with open("cities.csv", newline="", encoding="utf-8") as f:    state_of = {row["city"]: row["state"] for row in csv.DictReader(f)}print("lookup:", state_of)# 3. Stream the big file through, adding the new columnFIELDS = ["name", "city", "subject", "mark", "state"]with open("marks.csv", newline="", encoding="utf-8") as src, \     open("merged.csv", "w", newline="", encoding="utf-8") as dst:    reader = csv.DictReader(src)    writer = csv.DictWriter(dst, fieldnames=FIELDS)    writer.writeheader()    for row in reader:        row["state"] = state_of.get(row["city"], "unknown")   # .get, not [ ]        writer.writerow(row)print(open("merged.csv", encoding="utf-8").read())# Two design points worth naming:#  - The lookup file is loaded ONCE into a dict. Re-reading it per row would#    turn a fast job into a slow one.#  - .get(city, "unknown") means one missing city does not crash the whole#    merge - Day 5's safe-access lesson.## This is exactly what pd.merge() does for you in Module 1.17.

---## Done? Self-check- [ ] I know why `with` is better than calling `close()` myself- [ ] I can name what `"w"` does to an existing file- [ ] I know why `line == "Ravi"` fails when reading a file- [ ] I know that `write()` does not add a newline- [ ] I can say why `split(",")` is not safe on a CSV- [ ] I know every CSV value arrives as a string- [ ] I know why CSVs are opened with `newline=""`### Homework1. Write a word-frequency counter that reads a text file.2. Read a CSV and write out only the rows that pass a filter.3. Add `try`/`except` to every file function you have written.### Next class — Topic 1.10: Exception Handling & Modules`try` / `except` / `finally`, raising and writing your own exceptions, importing modulesand packages.---*Slides & notebooks by Srinivasa Sai Chava · Boston University*

---## Wrap-up — running the last 5 minutesThree cold-call questions:1. *"What does `"w"` do to a file that already exists?"* → erases it, instantly, no warning.2. *"Why does `line == "Ravi"` fail?"* → the trailing newline. Use `.strip()`.3. *"What type is `row["mark"]` from a CSV?"* → a string. Cast it.**Common misconceptions to watch for today**| Misconception | Correction ||---|---|| "`close()` is optional" | It is required; `with` does it for you, even on an error || "`"w"` appends if the file exists" | It erases the file completely || "Lines come back clean" | They keep their `\n`; the last one often does not || "`write()` adds a newline" | It does not. `print(..., file=f)` does || "`split(",")` reads a CSV" | It breaks on commas inside quoted fields || "CSV numbers are numbers" | Every value is a string until you cast it || "A file can be read twice in one `with`" | It is exhausted after one pass |**The habit worth naming: cast at the door.** Convert types the moment data enters theprogram, so nothing downstream has to wonder whether `mark` is a string. Students who dothis have far fewer mysterious `TypeError`s in Module 3.**Safety note:** if anyone overwrote a file during practice, use it. It is the mostmemorable lesson of the session, and the fix — write to a new filename — is a habit thatprotects them for the rest of their career.**Homework given:** word-frequency counter; filtered CSV; `try`/`except` on every file function.**Next session:** Topic 1.10 — Exception Handling & Modules. Today's `try`/`except` was apreview; tomorrow covers it properly, including writing your own exception types.